# Titanic ETL Pipeline
### Extract • Transform • Validate • Load • Batch Processing

**Project objective:** Build an ETL pipeline using Python, Pandas, Seaborn, SQLAlchemy and MySQL. The notebook combines the trainer's separate Python modules into one step-by-step, executable demonstration.

**Dataset:** Titanic dataset loaded through Seaborn (891 rows and 15 columns).



## 1. Install and import required libraries
Run the installation command once if these packages are not already installed.

In [1]:
%pip install pandas seaborn sqlalchemy mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [12]:
import argparse
import getpass
import pandas as pd
import seaborn as sns
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. MySQL database configuration
Enter your local MySQL details. The password is requested privately and is not written into the notebook.

In [3]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": getpass.getpass("Enter your MySQL password: "),
    "database": "titanic_db"
}

print("Database configuration prepared.")

Database configuration prepared.


## 3. Test MySQL connection
Create the database `titanic_db` in MySQL Workbench before running this section.

In [4]:
connection_url = URL.create(
    drivername="mysql+mysqlconnector",
    username=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    database=DB_CONFIG["database"]
)

engine = create_engine(connection_url)

try:
    with engine.connect() as connection:
        result = connection.execute(
            text("SELECT DATABASE(), VERSION()")
        ).fetchone()
        print("Connected successfully!")
        print("Database:", result[0])
        print("MySQL version:", result[1])
finally:
    engine.dispose()

Connected successfully!
Database: titanic_db
MySQL version: 8.0.46


## 4. Extract the Titanic dataset
Load the dataset and inspect its dimensions, first rows and missing values.

In [5]:
def extract():
    df = sns.load_dataset("titanic")
    print(f"Extracted shape: {df.shape}")
    return df

raw = extract()
display(raw.head())
display(raw.isnull().sum().to_frame("Missing values"))

Extracted shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


,Missing values
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


## 5. Transform the dataset
The transformation follows the trainer's project steps: fill missing age and embarked values, remove `deck` and `embark_town`, and encode `sex` as male = 0 and female = 1.

In [6]:
def transform(df):
    df = df.copy()
    df["age"] = df["age"].fillna(df["age"].median())
    df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
    df = df.drop(columns=["deck", "embark_town"])
    df["sex"] = df["sex"].map({"male": 0, "female": 1})
    print(f"Transformed shape: {df.shape}")
    return df

clean = transform(raw)
display(clean.head())
print("Missing values after transformation:")
display(clean.isnull().sum().to_frame("Missing values"))

Transformed shape: (891, 13)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,alive,alone
0,0,3,0,22.0,1,0,7.2500,S,Third,man,True,no,False
1,1,1,1,38.0,1,0,71.2833,C,First,woman,False,yes,False
2,1,3,1,26.0,0,0,7.9250,S,Third,woman,False,yes,True
3,1,1,1,35.0,1,0,53.1000,S,First,woman,False,yes,False
4,0,3,0,35.0,0,0,8.0500,S,Third,man,True,no,True


Missing values after transformation:


,Missing values
survived,0
pclass,0
sex,0
age,0
sibsp,0
parch,0
fare,0
embarked,0
class,0
who,0


## 6. Validate the transformed data
Check that no missing values remain and that the `sex` column contains only 0 or 1.

In [7]:
def validate(df):
    assert df.isnull().sum().sum() == 0, "Nulls still present!"
    assert df["sex"].isin([0, 1]).all(), "Sex not fully encoded!"
    print("Validation passed")

validate(clean)

Validation passed


## 7. Load data into MySQL
The cleaned dataset is loaded into the `passengers` table. Running this section again replaces the existing table.

In [8]:
def load(df, table_name="passengers", if_exists="replace"):
    engine = create_engine(connection_url)
    try:
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists=if_exists,
            index=False
        )
        print(f"Successfully loaded {len(df)} rows into '{table_name}'.")
    finally:
        engine.dispose()

load(clean)

Successfully loaded 891 rows into 'passengers'.


## 8. Run the complete normal ETL pipeline
This section runs extraction, transformation, validation and loading in sequence.

In [9]:
def run_pipeline():
    print("\n========== ETL PIPELINE STARTED ==========\n")

    print("STEP 1: Extracting data...")
    raw_data = extract()

    print("\nSTEP 2: Transforming data...")
    clean_data = transform(raw_data)

    print("\nSTEP 3: Validating data...")
    validate(clean_data)

    print("\nSTEP 4: Loading data into MySQL...")
    load(clean_data)

    print("\n========== ETL PIPELINE COMPLETED ==========\n")
    return clean_data

# Uncomment the next line to run the complete pipeline again:
# pipeline_result = run_pipeline()

## 9. Batch ETL processing
The dataset is processed in chunks of 100 rows. The first batch replaces the destination table; later batches append rows.

In [10]:
def run_batch_pipeline(batch_size=100, table_name="passengers_batch"):
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")

    raw_data = extract()
    total_rows = len(raw_data)

    if total_rows == 0:
        print("No rows found; nothing to load.")
        return

    print(f"Starting batch ETL: {total_rows} rows, batch size {batch_size}")
    loaded_rows = 0

    for batch_number, start in enumerate(
        range(0, total_rows, batch_size), start=1
    ):
        batch = raw_data.iloc[start:start + batch_size].copy()
        print(
            f"\nProcessing batch {batch_number}: "
            f"rows {start + 1}-{start + len(batch)}"
        )

        clean_batch = transform(batch)
        validate(clean_batch)

        mode = "replace" if batch_number == 1 else "append"
        load(clean_batch, table_name=table_name, if_exists=mode)
        loaded_rows += len(clean_batch)

    print(
        f"\nBatch ETL completed. Loaded {loaded_rows} "
        f"rows into '{table_name}'."
    )

# Run batch processing:
run_batch_pipeline(batch_size=100, table_name="passengers_batch")

Extracted shape: (891, 15)
Starting batch ETL: 891 rows, batch size 100

Processing batch 1: rows 1-100
Transformed shape: (100, 13)
Validation passed
Successfully loaded 100 rows into 'passengers_batch'.

Processing batch 2: rows 101-200
Transformed shape: (100, 13)
Validation passed
Successfully loaded 100 rows into 'passengers_batch'.

Processing batch 3: rows 201-300
Transformed shape: (100, 13)
Validation passed
Successfully loaded 100 rows into 'passengers_batch'.

Processing batch 4: rows 301-400
Transformed shape: (100, 13)
Validation passed
Successfully loaded 100 rows into 'passengers_batch'.

Processing batch 5: rows 401-500
Transformed shape: (100, 13)
Validation passed
Successfully loaded 100 rows into 'passengers_batch'.

Processing batch 6: rows 501-600
Transformed shape: (100, 13)
Validation passed
Successfully loaded 100 rows into 'passengers_batch'.

Processing batch 7: rows 601-700
Transformed shape: (100, 13)
Validation passed
Successfully loaded 100 rows into 'pass

## 10. Verify the data in MySQL
Run the SQL below in this notebook to check the row counts and preview records. You can also run the same SQL in MySQL Workbench.

In [11]:
verify_engine = create_engine(connection_url)

try:
    with verify_engine.connect() as connection:
        normal_count = connection.execute(
            text("SELECT COUNT(*) FROM passengers")
        ).scalar()

        batch_count = connection.execute(
            text("SELECT COUNT(*) FROM passengers_batch")
        ).scalar()

        print("Normal pipeline rows:", normal_count)
        print("Batch pipeline rows:", batch_count)

        preview = pd.read_sql(
            text("SELECT * FROM passengers LIMIT 10"),
            connection
        )
        display(preview)
finally:
    verify_engine.dispose()

Normal pipeline rows: 891
Batch pipeline rows: 891


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,alive,alone
0,0,3,0,22.0,1,0,7.2500,S,Third,man,1,no,0
1,1,1,1,38.0,1,0,71.2833,C,First,woman,0,yes,0
2,1,3,1,26.0,0,0,7.9250,S,Third,woman,0,yes,1
3,1,1,1,35.0,1,0,53.1000,S,First,woman,0,yes,0
4,0,3,0,35.0,0,0,8.0500,S,Third,man,1,no,1
5,0,3,0,28.0,0,0,8.4583,Q,Third,man,1,no,1
6,0,1,0,54.0,0,0,51.8625,S,First,man,1,no,1
7,0,3,0,2.0,3,1,21.0750,S,Third,child,0,no,0
8,1,3,1,27.0,0,2,11.1333,S,Third,woman,0,yes,0
9,1,2,1,14.0,1,0,30.0708,C,Second,child,0,yes,0


### SQL queries for MySQL Workbench

```sql
USE titanic_db;

SHOW TABLES;

SELECT * FROM passengers LIMIT 10;

SELECT COUNT(*) AS total_passengers
FROM passengers;

SELECT COUNT(*) AS total_batch_passengers
FROM passengers_batch;
```

**Expected row counts after successful execution:** 891 in `passengers` and 891 in `passengers_batch`.

## 11. Conclusion

This project demonstrates a complete ETL workflow using Python and MySQL:

- Extracted the Titanic dataset.
- Cleaned and transformed missing values and categorical data.
- Validated the transformed dataset.
- Loaded records into MySQL.
- Implemented batch processing and checked the resulting row counts.

**Note about batch processing:** The trainer's batch approach calculates the age median and embarked mode separately inside each batch. Therefore, imputed values can differ from those in the normal full-dataset pipeline, even though both tables contain 891 rows.